# CAD App GUI usig GRADIO

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
import plotly.io as pio
import pickle

sns.set_style(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pio.renderers.default = "vscode"
pd.set_option('display.max_columns', None)

In [2]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bound_ = {}
        self.upper_bound_ = {}

    def fit(self, X, y=None):
        
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bound_[col] = Q1 - (self.factor * IQR)
            self.upper_bound_[col] = Q3 + (self.factor * IQR)
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        X_capped = X.copy()
        for col in X.columns:
            if col in self.lower_bound_:
                X_capped[col] = np.clip(X_capped[col], self.lower_bound_[col], self.upper_bound_[col])
        return X_capped.values 

print("✅'OutlierCapper' defined!")

✅'OutlierCapper' defined!


In [3]:
# imporing models
xgb_clf = pickle.load(open("./models/xgb_clf.pkl", "rb"))
scaler_clust = pickle.load(open("./models/scaler_clust.pkl", "rb"))
kmeans = pickle.load(open("./models/kmeans.pkl", "rb"))

In [4]:
def add_cluster_features(patient_df):

    cluster_features_scaled = scaler_clust.transform(patient_df)

    cluster = kmeans.predict(cluster_features_scaled)[0]

    patient_df["cluster0"] = int(cluster == 0)
    patient_df["cluster1"] = int(cluster == 1)
    patient_df["cluster2"] = int(cluster == 2)

    return patient_df

In [5]:
def predict_cad(
    age, sex, bmi,
    current_smoker, ex_smoker, fh,
    bp, pr,
    edema, weak_peripheral_pulse,
    lung_rales, systolic_murmur, diastolic_murmur,
    typical_chest_pain, dyspnea,
    function_class,
    atypical,
    q_wave, st_elevation, st_depression,
    tinversion, lvh, poor_r_progression,
    fbs, cr, tg, ldl, hdl, bun, esr,
    ef_tte,
    region_rwma,
    vhd
):


    sex_map = {
        "Male": 0,
        "Female": 1
    }

    yes_no_map = {
        "No": 0,
        "Yes": 1
    }

    vhd_map = {
        "Normal": 0,
        "Mild": 1,
        "Moderate": 3,
        "Severe": 4
    }


    patient_df = pd.DataFrame([{
        "age": age,
        "sex": sex_map[sex],
        "bmi": bmi,

        "current smoker": yes_no_map[current_smoker],
        "ex-smoker": yes_no_map[ex_smoker],
        "fh": yes_no_map[fh],

        "bp": bp,
        "pr": pr,

        "edema": yes_no_map[edema],
        "weak peripheral pulse": yes_no_map[weak_peripheral_pulse],
        "lung rales": yes_no_map[lung_rales],
        "systolic murmur": yes_no_map[systolic_murmur],
        "diastolic murmur": yes_no_map[diastolic_murmur],

        "typical chest pain": yes_no_map[typical_chest_pain],
        "dyspnea": yes_no_map[dyspnea],

        "function class": function_class,

        "atypical": yes_no_map[atypical],

        "q wave": yes_no_map[q_wave],
        "st elevation": yes_no_map[st_elevation],
        "st depression": yes_no_map[st_depression],
        "tinversion": yes_no_map[tinversion],
        "lvh": yes_no_map[lvh],
        "poor r progression": yes_no_map[poor_r_progression],

        "fbs": fbs,
        "cr": cr,
        "tg": tg,
        "ldl": ldl,
        "hdl": hdl,
        "bun": bun,
        "esr": esr,

        "ef-tte": ef_tte,

        "region rwma": region_rwma,

        "vhd": vhd_map[vhd]
    }])

    patient_df = add_cluster_features(patient_df)

    y_proba = xgb_clf.predict_proba(patient_df)[:, 1] 

    prediction = int(y_proba >= 0.13)

    probability = float(y_proba [0])

    if prediction == 1:
        diagnosis = "CAD Detected"
    else:
        diagnosis = "No CAD Detected"

    return (
        diagnosis,
        f"{probability:.2%}",
        patient_df
    )


In [7]:
import gradio as gr

demo = gr.Interface(
    fn=predict_cad,

    inputs=[

        gr.Number(label="Age"),

        gr.Dropdown(
            ["Male", "Female"],
            label="Sex"
        ),

        gr.Number(label="BMI"),

        gr.Dropdown(["No", "Yes"], label="Current Smoker"),
        gr.Dropdown(["No", "Yes"], label="Ex-Smoker"),
        gr.Dropdown(["No", "Yes"], label="Family History"),

        gr.Number(label="Blood Pressure"),
        gr.Number(label="Pulse Rate"),

        gr.Dropdown(["No", "Yes"], label="Edema"),
        gr.Dropdown(["No", "Yes"], label="Weak Peripheral Pulse"),
        gr.Dropdown(["No", "Yes"], label="Lung Rales"),
        gr.Dropdown(["No", "Yes"], label="Systolic Murmur"),
        gr.Dropdown(["No", "Yes"], label="Diastolic Murmur"),

        gr.Dropdown(["No", "Yes"], label="Typical Chest Pain"),
        gr.Dropdown(["No", "Yes"], label="Dyspnea"),

        gr.Dropdown(
            [1, 2, 3, 4],
            label="CCS Function Class"
        ),

        gr.Dropdown(["No", "Yes"], label="Atypical Chest Pain"),

        gr.Dropdown(["No", "Yes"], label="Q Wave"),
        gr.Dropdown(["No", "Yes"], label="ST Elevation"),
        gr.Dropdown(["No", "Yes"], label="ST Depression"),
        gr.Dropdown(["No", "Yes"], label="T Wave Inversion"),
        gr.Dropdown(["No", "Yes"], label="LVH"),
        gr.Dropdown(["No", "Yes"], label="Poor R Progression"),

        gr.Number(label="Fasting Blood Sugar (mg/dL)"),
        gr.Number(label="Creatinine (mg/dL)"),
        gr.Number(label="Triglycerides (mg/dL)"),
        gr.Number(label="LDL (mg/dL)"),
        gr.Number(label="HDL (mg/dL)"),
        gr.Number(label="BUN (mg/dL)"),
        gr.Number(label="ESR (mm/h)"),

        gr.Number(label="Ejection Fraction (%)"),

        gr.Dropdown(
            [0, 1, 2, 3, 4],
            label="Regional Wall Motion Abnormality"
        ),

        gr.Dropdown(
            ["Normal", "Mild", "Moderate", "Severe"],
            label="Valvular Heart Disease"
        )

    ],

    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Textbox(label="CAD Probability"),
        gr.Dataframe(label="Model Input Data")
    ],

    title="Coronary Artery Disease Prediction",
    description="Prediction using XGBoost."
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
